# 🌦️ Análisis de Datos Meteorológicos en Tiempo Real (OpenWeather)

Este notebook analiza los datos ingestados periódicamente mediante **Node-RED** desde la API de **OpenWeather** y almacenados en **MongoDB Atlas**.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient
from dotenv import load_dotenv
from datetime import datetime

# Configuración de estilo
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Cargar variables de entorno
load_dotenv(dotenv_path="../../.env")
MONGO_URI = os.getenv("MONGO_URI")

## 1. Conexión a MongoDB Atlas

In [ ]:
try:
    client = MongoClient(MONGO_URI)
    db = client["airvlc_db"]
    collection = db["meteo_realtime"]
    print(f"✅ Conectado a MongoDB Atlas. Documentos actuales: {collection.count_documents({})}")
except Exception as e:
    print(f"❌ Error: {e}")

## 2. Carga de Datos en Pandas

In [ ]:
# Traer los últimos 1000 registros
cursor = collection.find().sort("timestamp", -1).limit(1000)
df = pd.DataFrame(list(cursor))

if not df.empty:
    # Convertir timestamp a objeto datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    # Eliminar la columna de ID de MongoDB para mayor claridad
    df.drop(columns=['_id'], inplace=True, errors='ignore')
    print("📊 Primeras filas cargadas:")
    display(df.head())
else:
    print("⚠️ La colección está vacía. ¿Has ejecutado el flujo de Node-RED?")

## 3. Análisis Descriptivo

In [ ]:
if not df.empty:
    display(df[['temp', 'humidity', 'pressure', 'wind_speed']].describe())
else:
    print("Sin datos para analizar.")

## 4. Visualización Temporal

In [ ]:
if not df.empty:
    fig, ax1 = plt.subplots()

    color = 'tab:red'
    ax1.set_xlabel('Tiempo')
    ax1.set_ylabel('Temperatura (°C)', color=color)
    ax1.plot(df['timestamp'], df['temp'], color=color, marker='o', label='Temp')
    ax1.tick_params(axis='y', labelcolor=color)

    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('Humedad (%)', color=color)
    ax2.plot(df['timestamp'], df['humidity'], color=color, linestyle='--', label='Humedad')
    ax2.tick_params(axis='y', labelcolor=color)

    plt.title('Evolución de Temperatura y Humedad en Valencia (Tiempo Real)')
    fig.tight_layout()
    plt.show()
else:
    print("Sin datos para graficar.")

## 5. Correlación de Variables Meteo

In [ ]:
if not df.empty:
    sns.heatmap(df[['temp', 'humidity', 'pressure', 'wind_speed']].corr(), annot=True, cmap='coolwarm', fmt=".2f")
    plt.title("Matriz de Correlación de Variables Meteorológicas")
    plt.show()
else:
    print("Sin datos para analizar correlación.")